# 01 — GRACE B2: cache đã chuẩn bị → retrieval → prompt

Chạy bằng Python trong WSL/Linux. Dữ liệu raw và cache CodeT5 nằm trong `output/local_full/<dataset>/`; CPG mặc định là snapshot đã phục hồi `cpg_recovered_joern_final_20261031`; không phụ thuộc GRACE-native ở home. Cache Devign dùng train native + validation/test Kaggle; ReVeal dùng cache đã xác minh.

Để xuất prompt: chạy cell thiết lập, rồi bước 3. Không gọi LLM trong notebook này. Các bước tải raw và trích Joern chỉ chạy khi bật cờ tương ứng; mặc định giữ dữ liệu đã chuẩn bị.


In [1]:
from pathlib import Path
import subprocess, sys, os, json
assert os.name != 'nt', 'Chọn kernel Python trong WSL/Linux'
search = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
ROOT = next((p for base in search for p in (base, base / '01_Author_Baseline_Unified')
             if (p / 'prepare_kaggle_bundle.py').is_file() and (p / 'evaluation_contract.yaml').is_file()), None)
if ROOT is None:
    raise FileNotFoundError(f'Không tìm thấy GRACE-summary/01_Author_Baseline_Unified từ {Path.cwd()}')
DATASETS = ['devign', 'reveal']
SPLITS = ['train', 'validation', 'test']
WORK = ROOT / 'output' / 'local_full'
CPG_ROOT = WORK / 'cpg_recovered_joern_final_20261031'
BUNDLE_ROOT = WORK / 'bundles_recovered_joern_final_20261031'
DOWNLOAD_RAW = False
REBUILD_CPG = False
JOERN_HOME = Path(os.environ.get('JOERN_HOME', '~/.cache/grace-h2/joern-cli')).expanduser().resolve()
print('Root:', ROOT)
print('Raw và embedding cache:', WORK)
print('CPG snapshot:', CPG_ROOT)
print('Bundle output:', BUNDLE_ROOT)


Root: /mnt/d/JUNIOR/LabThayTho/GRACE-summary/01_Author_Baseline_Unified
Raw và embedding cache: /mnt/d/JUNIOR/LabThayTho/GRACE-summary/01_Author_Baseline_Unified/output/local_full
CPG snapshot: /mnt/d/JUNIOR/LabThayTho/GRACE-summary/01_Author_Baseline_Unified/output/local_full/cpg_recovered_joern_final_20261031
Bundle output: /mnt/d/JUNIOR/LabThayTho/GRACE-summary/01_Author_Baseline_Unified/output/local_full/bundles_recovered_joern_final_20261031


## 1. Dữ liệu raw theo contract (đã có)

Mặc định bỏ qua download; chỉ kiểm tra đủ file. Đặt `DOWNLOAD_RAW=True` nếu cần tải lại theo revision contract.


In [2]:
for dataset in DATASETS:
    raw = WORK / dataset / 'raw'
    if DOWNLOAD_RAW:
        subprocess.run([sys.executable, str(ROOT / 'download_contract_data.py'), '--dataset', dataset, '--out', str(raw)], cwd=ROOT, check=True)
    for split in SPLITS:
        assert (raw / f'{split}.jsonl').is_file(), f'Thiếu raw: {dataset}:{split}'
    print('Raw ready:', dataset)


Raw ready: devign
Raw ready: reveal


## 2. CPG Joern (đã có)

Mặc định không chạy lại Joern. Chỉ cần cài Joern nếu bật `REBUILD_CPG=True`; bước dựng prompt không cần Joern CLI. Snapshot mặc định còn 240 graph thiếu theo audit; `--allow-missing-graphs` giữ các mẫu này trong bundle. Để trích lại Joern, chọn một `CPG_ROOT` mới trước khi bật `REBUILD_CPG`.


In [3]:
if REBUILD_CPG:
    assert CPG_ROOT != WORK / 'cpg_recovered_joern_final_20261031', 'Chọn CPG_ROOT mới trước khi bật REBUILD_CPG'
    for executable in ('joern-parse', 'joern-export'):
        file = JOERN_HOME / executable
        assert file.is_file() and os.access(file, os.X_OK), f'Joern không khả dụng: {file}'
for dataset in DATASETS:
    raw = WORK / dataset / 'raw'
    cpg = CPG_ROOT / dataset / 'cpg'
    for split in SPLITS:
        if REBUILD_CPG:
            cpg.mkdir(parents=True, exist_ok=True)
            subprocess.run([
                sys.executable, str(ROOT / 'prepare_joern_cpg.py'),
                '--input', str(raw / f'{split}.jsonl'), '--output', str(cpg / f'{split}.jsonl'),
                '--dataset', dataset, '--joern-home', str(JOERN_HOME), '--workers', '10', '--batch-size', '10'
            ], cwd=ROOT, check=True)
        assert (cpg / f'{split}.jsonl').is_file(), f'Thiếu CPG: {dataset}:{split}'
    print('CPG ready:', dataset)


CPG ready: devign
CPG ready: reveal


## 3. Đọc cache → tìm ứng viên → chọn demo → dựng prompt → ZIP

Không chạy lại CodeT5. Kiểm tra ID và hash mã nguồn CPG với cache trước khi dựng prompt. Cache tại `WORK/<dataset>/embedding_cache/{train,validation,test}/`. Giữ bước lọc donor theo cả hai tập đánh giá và fit whitening trên train hợp lệ. Xuất đầy đủ validation/test, không dùng LIMIT. Tokenizer Qwen tải theo revision contract; không gọi LLM.


In [4]:
# Check all six completed caches before starting any bundle.
sys.path.insert(0, str(ROOT))
from prepare_embedding_cache import CachedEmbedder, read_codes, code_hash
for dataset in DATASETS:
    cache = WORK / dataset / 'embedding_cache'
    for split in SPLITS:
        raw = WORK / dataset / 'raw' / f'{split}.jsonl'
        cpg_file = CPG_ROOT / dataset / 'cpg' / f'{split}.jsonl'
        assert cpg_file.is_file(), f'Thiếu CPG: {cpg_file}'
        rows = read_codes(raw)
        meta = json.loads((cache / split / 'manifest.json').read_text(encoding='utf-8'))
        expected = [{'id': key, 'code_sha256': code_hash(code)} for key, code in rows]
        assert meta['rows'] == expected and meta['completed'] == len(rows), f'Cache chưa hoàn tất/khác raw: {dataset}:{split}'
        expected_by_id = dict(rows)
        seen = set()
        with cpg_file.open(encoding='utf-8') as f:
            for line in f:
                if not line.strip():
                    continue
                item = json.loads(line)
                key = str(item['id'])
                assert key in expected_by_id and key not in seen, f'ID CPG thiếu hoặc trùng: {dataset}:{split}:{key}'
                assert code_hash(str(item['func']).strip()) == code_hash(expected_by_id[key]), f'Mã CPG khác raw/cache: {dataset}:{split}:{key}'
                seen.add(key)
        assert len(seen) == len(expected_by_id), f'CPG thiếu mẫu: {dataset}:{split}'
        print(f'CPG/cache identity: {dataset}/{split}: {len(seen)} rows', flush=True)
    loader = CachedEmbedder(cache)
    del loader
    print('Cache ready:', dataset, flush=True)

for dataset in DATASETS:
    cpg = CPG_ROOT / dataset / 'cpg'
    for split in ('validation', 'test'):
        out = BUNDLE_ROOT / dataset / split
        cmd = [sys.executable, str(ROOT / 'prepare_kaggle_bundle.py'), '--dataset', dataset, '--split', split,
               '--train', str(cpg / 'train.jsonl'), '--validation', str(cpg / 'validation.jsonl'),
               '--test', str(cpg / 'test.jsonl'), '--out', str(out), '--allow-missing-graphs',
               '--embedding-cache', str(WORK / dataset / 'embedding_cache')]
        print('RUN:', ' '.join(cmd), flush=True)
        subprocess.run(cmd, cwd=ROOT, check=True)
        print('Kaggle bundle:', out / f'{dataset}_{split}_b2_kaggle.zip', flush=True)


CPG/cache identity: devign/train: 21854 rows
CPG/cache identity: devign/validation: 2732 rows
CPG/cache identity: devign/test: 2732 rows
Cache ready: devign
CPG/cache identity: reveal/train: 18187 rows
CPG/cache identity: reveal/validation: 2273 rows
CPG/cache identity: reveal/test: 2274 rows
Cache ready: reveal
RUN: /mnt/d/JUNIOR/LabThayTho/GRACE-summary/.venv-wsl/bin/python /mnt/d/JUNIOR/LabThayTho/GRACE-summary/01_Author_Baseline_Unified/prepare_kaggle_bundle.py --dataset devign --split validation --train /mnt/d/JUNIOR/LabThayTho/GRACE-summary/01_Author_Baseline_Unified/output/local_full/cpg_recovered_joern_final_20261031/devign/cpg/train.jsonl --validation /mnt/d/JUNIOR/LabThayTho/GRACE-summary/01_Author_Baseline_Unified/output/local_full/cpg_recovered_joern_final_20261031/devign/cpg/validation.jsonl --test /mnt/d/JUNIOR/LabThayTho/GRACE-summary/01_Author_Baseline_Unified/output/local_full/cpg_recovered_joern_final_20261031/devign/cpg/test.jsonl --out /mnt/d/JUNIOR/LabThayTho/GRACE

Các ZIP nằm tại output/local_full/bundles_recovered_joern_final_20261031/<dataset>/<split>/. Upload bundle tương ứng thành Kaggle Dataset riêng, rồi mở notebook 02. Notebook 02 vẫn mặc định smoke 20 mẫu.
